# Tiền xử lý dữ liệu

### Import thư viện

In [99]:
# Data processing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display plots in notebook
%matplotlib inline

# Regex
import re

# Path
import os

# Split data and scale features
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Data directory
DATA_DIR = os.path.join(os.getcwd(), "..", "..", "data")
print("Data directory:", DATA_DIR)

Data directory: c:\Users\ADMIN\OneDrive\Desktop\Housing-Market-Prediction\Nhom20\notebooks\Model_Comparison\..\..\data


## 1. Nạp dữ liệu

In [100]:
df = pd.read_csv(os.path.join(DATA_DIR, "raw", "data_public.csv"))

# Show the first 5 rows of the dataframe
df.head()

,Title,Price,Area,Location,Listing ID,Last Updated,Property Type,Width,Length,Bedrooms,...,Longitude,VIP Account,Avatar,Agent Role,Agent Name,Agent Listing Count,Province,Property Type Slug,Scraped At,Last Updated Date
0,"🏠 NHÀ 3 TẦNG – TRUNG TÂM PHƯỚC LONG B, THỦ ĐỨC",7500.0,95.0,"Phường Phước Long,TP. Hồ Chí Minh",1546393.0,2 giờ trước,Nhà riêng,5.0,17.0,3.0,...,NaN,False,1,Môi giới,Ngô Quang Tùng,20.0,tp-ho-chi-minh,nha-mat-pho-mat-tien,2025-09-30 18:24:18,30/09/2025 16:24
1,🔥🔥 NHÀ ĐẸP HIỆP BÌNH PHƯỚC – HẺM XE HƠI – KHU ...,4500.0,66.0,"đường Hiệp Bình,Phường Hiệp Bình Chánh,Quận Th...",1528310.0,2 giờ trước,Nhà riêng,4.0,NaN,2.0,...,NaN,False,1,Môi giới,Ngô Quang Tùng,20.0,tp-ho-chi-minh,nha-mat-pho-mat-tien,2025-09-30 18:24:18,30/09/2025 16:24
2,🌟 HXH - NHÀ 3 TẦNG ĐẸP – TRUNG TÂM PHƯỚC LONG ...,7500.0,92.0,"Phường Phước Long B,Quận 9,TP. Hồ Chí Minh",1546519.0,2 giờ trước,Nhà riêng,5.0,17.0,3.0,...,NaN,False,1,Môi giới,Ngô Quang Tùng,20.0,tp-ho-chi-minh,nha-mat-pho-mat-tien,2025-09-30 18:24:18,30/09/2025 16:24
3,🏡 HÀNG HIẾM LINH CHIỂU – NHÀ 2 TẦNG – VỊ TRÍ V...,4750.0,45.0,"đường số 19,Phường Linh Chiểu,Quận Thủ Đức,TP....",1544240.0,2 giờ trước,Nhà riêng,4.0,11.5,3.0,...,NaN,False,1,Môi giới,Ngô Quang Tùng,20.0,tp-ho-chi-minh,nha-mat-pho-mat-tien,2025-09-30 18:24:18,30/09/2025 16:24
4,🚗 NHÀ ĐẸP HẺM XE HƠI – VỊ TRÍ VIP LINH TÂY – G...,4800.0,82.0,"đường số 9,Phường Linh Tây,Quận Thủ Đức,TP. Hồ...",1528291.0,2 giờ trước,Nhà riêng,8.5,NaN,2.0,...,NaN,False,1,Môi giới,Ngô Quang Tùng,20.0,tp-ho-chi-minh,nha-mat-pho-mat-tien,2025-09-30 18:24:18,30/09/2025 16:24


## 2. Dọn dẹp dữ liệu trùng và không hợp lệ

#### 2.1. Dọn dẹp dữ liệu không hợp lệ

In [101]:
print("Dataset shape before removing invalid rows:", df.shape)

# Drop rows with non-positive price or area
df = df[
    (df["Price"] > 0) &
    (df["Area"] > 0)
]

print("Dataset shape after removing invalid rows:", df.shape)

Dataset shape before removing invalid rows: (51304, 28)
Dataset shape after removing invalid rows: (51182, 28)


#### 2.2. Dọn dẹp dữ liệu trùng

In [102]:
# Count duplicate rows
duplicate_count = df.duplicated().sum()

print("Number of duplicated rows:", duplicate_count)

# Remove duplicate rows and reset index
df = df.drop_duplicates()

duplicate_count_after = df.duplicated().sum()

print("Number of duplicated rows after dropping duplicates:", duplicate_count_after)

df = df.reset_index(drop=True)

Number of duplicated rows: 0
Number of duplicated rows after dropping duplicates: 0


## 3. Chia tập dữ liệu train/test

In [103]:
# Split train/test
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

# Reset index
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (40945, 28)
Test shape : (10237, 28)


## 4. Clipping 1st và 99th percentiles cho các numeric columns

In [104]:
numeric_cols = [
    "Area",
    "Width",
    "Length",
    "Bedrooms",
    "Bathrooms",
    "Floors",
    "Alley Width"
]

print("Before clipping: ")
display(train_df[numeric_cols].describe().T[["min", "max"]])

# Calculate 1st and 99th percentiles for numeric columns
print("\nClipping numeric columns to 1st and 99th percentiles:")
clip_dict = {}

for col in numeric_cols:

    lower = train_df[col].quantile(0.01)
    upper = train_df[col].quantile(0.99)

    clip_dict[col] = (lower, upper)

    print(
        f"{col}: "
        f"lower={lower:.2f}, "
        f"upper={upper:.2f}"
    )
    
# Clip numeric columns in train set    
for col in numeric_cols:

    lower, upper = clip_dict[col]

    train_df[col] = train_df[col].clip(
        lower=lower,
        upper=upper
    )
# Clip numeric columns in test set
for col in numeric_cols:

    lower, upper = clip_dict[col]

    test_df[col] = test_df[col].clip(
        lower=lower,
        upper=upper
    )
    
print("\nAfter clipping: ")
display(train_df[numeric_cols].describe().T[["min", "max"]])

Before clipping: 


,min,max
Area,1.0,1.578861e+09
Width,0.0,5.000000e+04
Length,0.0,9.000000e+04
Bedrooms,0.0,6.902000e+03
Bathrooms,0.0,1.100000e+02
Floors,-2.0,9.191688e+08
Alley Width,0.0,1.000000e+06



Clipping numeric columns to 1st and 99th percentiles:
Area: lower=32.00, upper=16279.18
Width: lower=3.22, upper=124.87
Length: lower=7.00, upper=177.00
Bedrooms: lower=1.00, upper=16.00
Bathrooms: lower=1.00, upper=12.00
Floors: lower=0.50, upper=8.00
Alley Width: lower=1.00, upper=60.00

After clipping: 


,min,max
Area,32.0000,16279.184
Width,3.2215,124.871
Length,7.0000,177.000
Bedrooms,1.0000,16.000
Bathrooms,1.0000,12.000
Floors,0.5000,8.000
Alley Width,1.0000,60.000


## 5. Xử lý outlier

### 5.1. Tạo đặc trưng price_per_m2 

In [105]:
# Create price_per_m2 feature
train_df["price_per_m2"] = (
    train_df["Price"]
    / train_df["Area"]
)

test_df["price_per_m2"] = (
    test_df["Price"]
    / test_df["Area"]
)

# Checking dataset
train_df["price_per_m2"].describe()

count    4.094500e+04
mean     3.443609e+02
std      8.688216e+03
min      1.625000e-03
25%      1.668405e+01
50%      4.761905e+01
75%      1.000000e+02
max      1.718750e+06
Name: price_per_m2, dtype: float64

### 5.2. Tạo đặc trưng District

In [106]:
# Create District feature by extracting from Location
train_df["District"] = train_df["Location"].str.extract(
    r"(Quận\s+\d+|Quận\s+[^\.,]+|Huyện\s+[^\.,]+|TP\.\s*[^\.,]+)"
)
train_df["District"].fillna("Unknown", inplace=True)

test_df["District"] = test_df["Location"].str.extract(
    r"(Quận\s+\d+|Quận\s+[^\.,]+|Huyện\s+[^\.,]+|TP\.\s*[^\.,]+)"
)
test_df["District"].fillna("Unknown", inplace=True)

print("Unique districts in train set:\n", train_df["District"].unique())
print("\nValues counts of districts in train set:\n")
display(train_df["District"].value_counts())
print("Unique districts in test set:\n", test_df["District"].unique())
print("\nValues counts of districts in test set:\n")
display(test_df["District"].value_counts())

Unique districts in train set:
 ['Huyện Nhà Bè' 'TP. Hồ Chí Minh(Mới)' 'Quận 12' 'Quận Thủ Đức'
 'Huyện Củ Chi' 'Quận 4' 'Quận Tân Phú' 'Quận 9' 'Huyện Bình Chánh'
 'Quận Bình Thạnh' 'Quận 7' 'Quận 8' 'Huyện Hóc Môn' 'Quận 6'
 'TP. Hồ Chí Minh' 'Quận 3' 'Quận 5' 'Huyện Cần Giờ' 'Quận 10'
 'Quận Gò Vấp' 'Quận 1' 'Quận Bình Tân' 'Quận 11' 'Quận Tân Bình'
 'Unknown' 'Quận Phú Nhuận' 'Quận 2' 'Quận Thủ Đức (Cũ)'
 'Huyện Thanh Quan']

Values counts of districts in train set:



District
TP. Hồ Chí Minh(Mới)    17094
Quận 9                   3282
Quận 7                   2861
Quận 12                  2194
Quận Thủ Đức             1592
Huyện Củ Chi             1529
Huyện Bình Chánh         1514
TP. Hồ Chí Minh          1459
Quận Bình Tân            1188
Quận Bình Thạnh          1120
Quận Tân Phú             1044
Huyện Hóc Môn             991
Huyện Nhà Bè              908
Quận 1                    688
Quận Gò Vấp               589
Quận Tân Bình             438
Quận 8                    421
Quận 4                    403
Quận 6                    292
Quận 10                   288
Quận 3                    231
Quận Phú Nhuận            217
Quận 11                   168
Huyện Cần Giờ             167
Quận 5                    115
Quận 2                    113
Quận Thủ Đức (Cũ)          22
Unknown                    10
Huyện Thanh Quan            7
Name: count, dtype: int64

Unique districts in test set:
 ['Quận Thủ Đức' 'Quận Bình Thạnh' 'Quận 5' 'TP. Hồ Chí Minh(Mới)' 'Quận 7'
 'Quận 12' 'Quận Tân Phú' 'Huyện Cần Giờ' 'Huyện Củ Chi' 'Quận Bình Tân'
 'Huyện Hóc Môn' 'Quận Tân Bình' 'Quận 8' 'Quận 9' 'Huyện Bình Chánh'
 'TP. Hồ Chí Minh' 'Quận 10' 'Quận 1' 'Huyện Nhà Bè' 'Quận 4'
 'Quận Gò Vấp' 'Quận 3' 'Quận 6' 'Quận 2' 'Quận 11' 'Quận Phú Nhuận'
 'Quận Thủ Đức (Cũ)' 'Huyện Ba Vì' 'Unknown' 'Huyện Thanh Quan']

Values counts of districts in test set:



District
TP. Hồ Chí Minh(Mới)    4278
Quận 9                   809
Quận 7                   739
Quận 12                  547
Quận Thủ Đức             406
Huyện Củ Chi             389
TP. Hồ Chí Minh          388
Huyện Bình Chánh         363
Quận Bình Tân            287
Quận Bình Thạnh          281
Quận Tân Phú             244
Huyện Nhà Bè             237
Huyện Hóc Môn            231
Quận 1                   179
Quận Gò Vấp              148
Quận Tân Bình            125
Quận 8                    99
Quận 4                    89
Quận 10                   75
Quận 6                    67
Quận Phú Nhuận            51
Quận 3                    50
Quận 11                   49
Huyện Cần Giờ             45
Quận 5                    27
Quận 2                    25
Quận Thủ Đức (Cũ)          3
Unknown                    3
Huyện Thanh Quan           2
Huyện Ba Vì                1
Name: count, dtype: int64

### 5.4. Tạo đặc trưng is_luxury

In [107]:
luxury_keywords = [
    "villa",
    "penthouse",
    "cao cấp",
    "mặt tiền",
    "view sông",
    "biệt thự",
    "resort"
]

# Create regex pattern
pattern = "|".join(luxury_keywords)

# Create is_luxury column
train_df["is_luxury"] = (
    (
        train_df["Title"].fillna("") + " " +
        train_df["Description"].fillna("")
    )
    .str.lower()
    .str.contains(pattern, regex=True)
    .astype(int)
)

test_df["is_luxury"] = (
    (
        test_df["Title"].fillna("") + " " +
        test_df["Description"].fillna("")
    )
    .str.lower()
    .str.contains(pattern, regex=True)
    .astype(int)
)

train_df[["Title", "is_luxury"]].head(20)

,Title,is_luxury
0,Cần bán ot sunrise riverside 2pn 1wc new 100% ...,0
1,Nhà Hiệp Thành Thủ Dầu Một Bình Dương cần bán ...,0
2,"Bán đất thổ cư đường thạnh xuân 51, phường thạ...",0
3,Bán lô đất giá đầu tư 110m2 đ. số 48 đường xe ...,0
4,Bán đất 565m² ngang 25m giá 2.65 tỷ Xã Hồ Tràm,0
5,Bán đất 107.8m² 1.6 tỷ tại Phường Kim Dinh Thà...,0
6,"Bán đất ở tại Củ Chi, TP HCM",0
7,Cần bán nhanh căn hộ quận 4 saigon royal 60m2 ...,0
8,"Mở bán các nền đất mặt tiền luỹ bán bích, phườ...",1
9,Bán nhà 76.1m² ngang 4m giá 11.9 tỷ Phường Tân...,0


### 5.5. Tính toán IQR và bounds của price_per_m2 theo từng quận/huyện

In [108]:
# Compute district-level IQR thresholds for price_per_m2 on the train set
district_bounds = (
    train_df.groupby("District")["price_per_m2"]
    .quantile([0.25, 0.75])
    .unstack(level=-1)
    .rename(columns={0.25: "q1", 0.75: "q3"})
 )
district_bounds["iqr"] = district_bounds["q3"] - district_bounds["q1"]
district_bounds["lower"] = district_bounds["q1"] - 1.5 * district_bounds["iqr"]
district_bounds["upper"] = district_bounds["q3"] + 1.5 * district_bounds["iqr"]
district_bounds = district_bounds.reset_index()

# Merge district bounds into train_df so we can count outliers per district
train_df = train_df.merge(
    district_bounds[["District", "lower", "upper"]],
    on="District",
    how="left"
)
train_df["price_per_m2_outlier"] = (
    (train_df["price_per_m2"] < train_df["lower"]) |
    (train_df["price_per_m2"] > train_df["upper"])
).astype(int)

test_df = test_df.merge(
    district_bounds[["District", "lower", "upper"]],
    on="District",
    how="left"
)
test_df["price_per_m2_outlier"] = (
    (test_df["price_per_m2"] < test_df["lower"]) |
    (test_df["price_per_m2"] > test_df["upper"])
).astype(int)


district_outlier_counts = (
    train_df.groupby("District")["price_per_m2_outlier"]
    .sum()
    .reset_index(name="outlier_count")
 )

district_bounds = district_bounds.merge(
    district_outlier_counts,
    on="District",
    how="left"
 )
district_bounds["outlier_count"] = district_bounds["outlier_count"].fillna(0).astype(int)

# Round numeric bound values for display
district_bounds[["q1", "q3", "iqr", "lower", "upper"]] = district_bounds[["q1", "q3", "iqr", "lower", "upper"]].round(2)
district_bounds.sort_values("outlier_count", ascending=False, inplace=True)
# Display IQR bounds and outlier counts per district
print("District-level price_per_m2 IQR bounds and outlier counts:")
display(district_bounds[["District", "q1", "q3", "iqr", "lower", "upper", "outlier_count"]])

print("Train total outlier count:", train_df["price_per_m2_outlier"].sum())

District-level price_per_m2 IQR bounds and outlier counts:


,District,q1,q3,iqr,lower,upper,outlier_count
27,TP. Hồ Chí Minh(Mới),7.42,51.20,43.78,-58.25,116.87,1478
17,Quận 9,46.88,185.71,138.83,-161.36,393.96,752
9,Quận 12,37.31,91.72,54.41,-44.31,173.33,401
15,Quận 7,66.18,420.00,353.82,-464.56,950.74,362
0,Huyện Bình Chánh,17.46,58.31,40.85,-43.82,119.58,244
22,Quận Thủ Đức,59.49,113.79,54.31,-21.97,195.25,229
25,Quận Tân Phú,60.42,165.04,104.63,-96.53,321.99,222
2,Huyện Củ Chi,5.20,16.59,11.39,-11.88,33.67,138
4,Huyện Nhà Bè,33.06,72.24,39.18,-25.70,131.01,137
19,Quận Bình Tân,46.65,100.00,53.35,-33.36,180.02,124


Train total outlier count: 4882


### 5.6. Đánh dấu mẫu bị nghi ngờ và xóa bỏ

In [109]:
train_df["is_suspicious"] = (
    (train_df["price_per_m2_outlier"] == 1) &
    (train_df["is_luxury"] == 0)
).astype(int)
test_df["is_suspicious"] = (
    (test_df["price_per_m2_outlier"] == 1) &
    (test_df["is_luxury"] == 0)
).astype(int)

display(train_df["is_suspicious"].value_counts())

suspicious_df = train_df[train_df["is_suspicious"] == 1]
print("Suspicious listings (outliers that are not luxury):")
display(suspicious_df[["Title", "Description", "price_per_m2", "District", "is_luxury"]].head(20))

is_suspicious
0    38450
1     2495
Name: count, dtype: int64

Suspicious listings (outliers that are not luxury):


,Title,Description,price_per_m2,District,is_luxury
2,"Bán đất thổ cư đường thạnh xuân 51, phường thạ...","Bán đất thổ cư đường THẠNH XUÂN 51, phường THẠ...",3706.666667,Quận 12,0
3,Bán lô đất giá đầu tư 110m2 đ. số 48 đường xe ...,Bán lô đất đẹp hẻm đường Số 48 phường Hiệp Bìn...,6136.363636,Quận Thủ Đức,0
9,Bán nhà 76.1m² ngang 4m giá 11.9 tỷ Phường Tân...,NaN,156.373193,TP. Hồ Chí Minh(Mới),0
11,Căn hộ prosper plaza sổ hồng cầm tay nh hỗ trợ...,"TỔNG QUAN CÓ 1500 CĂN HỘTIỆN ÍCH: HỒ BƠ, SIÊU ...",4076.923077,Quận 12,0
21,"Đất bán ,hẻm xe hơi , khu phân lô, thạnh xuân ...",HẺM XE HƠI 7M KHU PHÂN LÔ GẦN PICITY - VỊ TRÍ ...,6517.857143,Quận 12,0
27,"Chủ kẹt tiền cần bán prosper plaza, phan văn h...","Prosper Plaza, Phan Văn Hớn, Tân Thới Nhất, Qu...",2369.230769,Quận 12,0
31,Đất đẹp long bình q9 ra vin q9 vài bước giá ch...,"Đất đẹp Long Bình Q9, giá chỉ 3.2 tỷ chính chủ...",524.590164,Quận 9,0
38,"Giảm 600tr! Nhà 5 tầng 40m², Vuông, Đ.6m – Phạ...","Giảm 600tr! Nhà 5 tầng 40m², Vuông, Đ.6m – Phạ...",132.250000,TP. Hồ Chí Minh(Mới),0
59,Bán nhà 100m² ngang 5m giá 15 tỷ tại Phường Tâ...,Hotline: 0986694438 🔸Ký Gửi - Mua Bán: Bất Độn...,150.000000,TP. Hồ Chí Minh(Mới),0
60,Bán lô đất đường thông diện tích khủng nguyễn ...,BÁN LÔ ĐẤT ĐƯỜNG THÔNG DIỆN TÍCH KHỦNG________...,601.851852,Quận 12,0


In [110]:
print("Train dataset shape before removing suspicious listings:", train_df.shape)

train_df = train_df[(train_df["is_suspicious"] == 0)].reset_index(drop=True)

print("Train dataset shape after removing suspicious listings:", train_df.shape)

print("Test dataset shape before removing suspicious listings:", test_df.shape)

test_df = test_df[(test_df["is_suspicious"] == 0)].reset_index(drop=True)

print("Test dataset shape after removing suspicious listings:", test_df.shape)

Train dataset shape before removing suspicious listings: (40945, 35)
Train dataset shape after removing suspicious listings: (38450, 35)
Test dataset shape before removing suspicious listings: (10237, 35)
Test dataset shape after removing suspicious listings: (9655, 35)


## 6. Xử lý missing values

### 6.1. Đánh dấu missing và fill median cho numeric columns

In [111]:
impute_cols = [
    "Area",
    "Width",
    "Length",
    "Bedrooms",
    "Bathrooms",
    "Floors",
    "Alley Width",
    "Agent Listing Count"
]

# Create missing indicator columns for imputed features
for col in impute_cols:

    # Train
    train_df[f"{col}_missing"] = (
        train_df[col]
        .isnull()
        .astype(int)
    )

    # Test
    test_df[f"{col}_missing"] = (
        test_df[col]
        .isnull()
        .astype(int)
    )

# Fill missing values with median of train set
for col in impute_cols:

    median_value = train_df[col].median()

    train_df[col] = train_df[col].fillna(
        median_value
    )

    test_df[col] = test_df[col].fillna(
        median_value
    )

### 6.2. Fill "Unknown" cho categorical columns

In [112]:
cat_cols = [
    "Direction",
    "Road Type",
    "Position",
    "Property Type",
    "Agent Role"
]

# Fill missing values with "Unknown"
for col in cat_cols:

    train_df[col] = train_df[col].fillna("Unknown")
    test_df[col] = test_df[col].fillna("Unknown")

## 7. One-hot encoding

In [113]:
# One-hot encode categorical columns
one_hot_cols = [
    "Direction",
    "Road Type",
    "Position",
    "Property Type",
    "Agent Role",
    "District",
]

print("Train shape before encoding:", train_df.shape)
print("Test shape before encoding:", test_df.shape)

train_df = pd.get_dummies(
    train_df,
    columns=one_hot_cols,
    prefix_sep="_",
    dtype=int,
)
test_df = pd.get_dummies(
    test_df,
    columns=one_hot_cols,
    prefix_sep="_",
    dtype=int,
)

# Align train and test columns to ensure both sets have the same features
train_df, test_df = train_df.align(test_df, join="outer", axis=1, fill_value=0)

print("One-hot encoding completed.")
print("Train shape after encoding:", train_df.shape)
print("Test shape after encoding:", test_df.shape)

Train shape before encoding: (38450, 43)
Test shape before encoding: (9655, 43)
One-hot encoding completed.
Train shape after encoding: (38450, 93)
Test shape after encoding: (9655, 93)


## 8. Standard Scaling

In [114]:
numeric_cols = [
    "Area",
    "Width",
    "Length",
    "Bedrooms",
    "Bathrooms",
    "Floors",
    "Alley Width"
]

# Standardize numeric features
scaler = StandardScaler()
train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])
print("Numeric features standardized.")

# Log transform the target variable
log_cols = ["Price", "Agent Listing Count"]
for col in log_cols:
    train_df[col] = np.log1p(train_df[col])
    test_df[col] = np.log1p(test_df[col])
print("Target variables logged.")

Numeric features standardized.
Target variables logged.


## 9. Drop columns không cần thiết

In [115]:
drop_cols = [
    "Listing ID",
    "Title",
    "Description",
    "Avatar",
    "Agent Name",
    "Last Updated",
    "Scraped At",
    "Last Updated Date",
    "Property Type Slug",
    "Location",
    "Province",
    "Latitude",
    "Longitude",
    "price_per_m2_outlier",
    "lower",
    "upper",
    "VIP Account",
    "price_per_m2",
    "is_suspicious"
]

train_df.drop(columns=drop_cols, inplace=True)
test_df.drop(columns=drop_cols, inplace=True)

print("Final train shape:", train_df.shape)
print("Final test shape :", test_df.shape)

Final train shape: (38450, 74)
Final test shape : (9655, 74)


## 10. Lưu dữ liệu train/test

In [116]:
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

train_df.to_csv(os.path.join(PROCESSED_DIR, "train_v2.csv"), index=False)
test_df.to_csv(os.path.join(PROCESSED_DIR, "test_v2.csv"), index=False)

print("Preprocessed datasets saved to:", PROCESSED_DIR)

Preprocessed datasets saved to: c:\Users\ADMIN\OneDrive\Desktop\Housing-Market-Prediction\Nhom20\notebooks\Model_Comparison\..\..\data\processed
